# P5: HUC-12 Land Use & BMP Merge

Primary merge per `MERGE.md` (§2, **P5**). Combines the HUC-12 **land-use**
fractions with the HUC-8 **conservation-BMP** adoption counts into one wide table
at **1 row per `huc12_code` + `year`**.

The two sources sit at different watershed resolutions, so the join is
`huc8_code = left(huc12_code, 8)` + `year`: each HUC-8's BMP values are
**broadcast to every child HUC-12** of that HUC-8. This is a repetition, not a
true sub-division of adoption within the HUC-8 (see `MERGE.md` §6) — worth
flagging in any BMP-effect analysis.

The BMP table is **pivoted wide** first — its `practice_type` × `unit` × category
descriptors become columns — so the `huc8_code` + `year` join can't fan the
land-use table out.

**Inputs** (both `data/tabular/02_clean/...`):
- `landuse/cdl-huc12-fractions-clean.csv` — CDL land-cover fractions, 1 row per
  `huc12_code` + `year` (2015–2025). Carried through unchanged; it is the base
  grain and drives the output row set.
- `bmp/iowa-nrs-bmp-huc8-clean.csv` — Iowa NRS practice-adoption counts, 1 row
  per `huc8_code` + `year` + `practice_type`, pivoted by practice.

**Output:** `data/03a_merge_primary/huc12-landuse-bmp.csv`, one row per
HUC-12 + year, carrying the derived `huc8_code` join key for the downstream S2
merge.

**Design notes**
- **Land use is the base (left) table.** The output row set is exactly the
  CDL HUC-12 × year grid; BMP is broadcast onto it. HUC-8 BMP years with no CDL
  HUC-12 (the pre-2015 history) are therefore dropped — they can't be expressed
  at HUC-12 grain anyway.
- **BMP carries two crep-wetland measures and a bioreactor vintage split.**
  `crep_wetland` is reported as both `Acres` and `Number`, so `unit` is part of
  the pivot key and both become distinct columns. `bioreactor_sat_buffer` has
  two parallel category series — the original and a *(Updated 2022)* re-count —
  that overlap in time; they are kept as separate `...__upd2022__` columns rather
  than silently averaged together.
- **Exact-year join, sparse by design.** BMP runs 2003–2022 while CDL runs
  2015–2025, so the BMP block is null for 2023–2025 (no BMP data yet). Cover-crop
  acreage is only published for 2017 and 2022, so that column is null in the
  other years. No carry-forward is applied here (unlike P3b) — the plan specifies
  an exact `year` join for P5.

In [1]:
import os

import pandas as pd

CLEAN = "../../data/tabular/02_clean"
OUT_DIR = "../../data/03a_merge_primary"
OUT_FILE = f"{OUT_DIR}/huc12-landuse-bmp.csv"

KEY = ["huc12_code", "year"]
HUC8_JOIN = ["huc8_code", "year"]

## Step 1: Land use (base table)

One row per `huc12_code` + `year`. HUC codes are read as strings to preserve the
leading zero (`070200090101`), and the 8-digit `huc8_code` join key is derived as
the first 8 characters of the HUC-12 code. This table defines the output grain,
so its `(huc12_code, year)` key is asserted unique up front.

**Censoring flags are reported and dropped here.** The cleaner
(`cdl-huc12-fractions-clean.ipynb`, Step 3) marks land-cover fractions that the
CDL source reported at its four-decimal limit — non-detects, where the true share
is only known to be below `5e-5` — and substitutes `LOD/2` for them, carrying one
`<col>_censored` boolean per fraction. Those flags are part of the `02_clean`
contract, not of the modeling table: this merge summarises them and then drops
them, so the downstream S2 → T1 → `epa-full.csv` column set is unchanged. Read
them from `02_clean/landuse/` if an analysis needs to know which values were
non-detects.

In [2]:
lu = pd.read_csv(f"{CLEAN}/landuse/cdl-huc12-fractions-clean.csv", dtype={"huc12_code": str})
lu["year"] = lu["year"].astype(int)
lu["huc8_code"] = lu["huc12_code"].str[:8]

assert lu["huc12_code"].str.len().eq(12).all(), "non-12-digit HUC-12 code present"
assert not lu.duplicated(subset=KEY).any(), "land-use base is not unique on (huc12_code, year)"

# Report the cleaner's non-detect flags, then drop them: they belong to the
# 02_clean contract, not to the modeling table (see the markdown above).
censor_cols = [c for c in lu.columns if c.endswith("_censored")]
if censor_cols:
    n_cells = int(lu[censor_cols].to_numpy().sum())
    n_rows = int(lu[censor_cols].any(axis=1).sum())
    print(f"land-cover non-detects: {n_cells:,} cells across {n_rows:,} rows "
          f"(carrying LOD/2); dropping {len(censor_cols)} *_censored flags")
    for c in censor_cols:
        n = int(lu[c].sum())
        if n:
            print(f"  {c:<28} {n:>5,}")
    lu = lu.drop(columns=censor_cols)

print(f"land use: {len(lu):,} rows | {lu['huc12_code'].nunique():,} HUC-12s x "
      f"years {lu['year'].min()}-{lu['year'].max()}")
lu.head(3)

land-cover non-detects: 282 cells across 273 rows (carrying LOD/2); dropping 9 *_censored flags
  pct_wetland_censored            97
  pct_open_water_censored        185
land use: 18,854 rows | 1,714 HUC-12s x years 2015-2025


,huc12_code,year,pct_corn,pct_soybean,pct_other_crops,pct_developed,pct_forest,pct_pasture,pct_wetland,pct_open_water,pct_row_crops,huc8_code
0,070200090101,2015,0.5396,0.3456,0.0051,0.0615,0.0009,0.0158,0.0155,0.0003,0.8852,07020009
1,070200090101,2016,0.5044,0.3823,0.0081,0.0614,0.0011,0.0271,0.0041,0.0002,0.8867,07020009
2,070200090101,2017,0.5585,0.3456,0.0017,0.0605,0.0007,0.0258,0.0058,0.0002,0.9041,07020009


## Step 2: BMP adoption, pivoted wide by practice

The BMP table is long: one row per `huc8_code` + `year` + `practice_type`, but
with two wrinkles that make `(huc8_code, year, practice_type)` **non-unique** on
its own — so pivoting on `practice_type` alone would silently aggregate:

- `crep_wetland` is reported in **two units** (`Acres` and `Number`) — wetland
  area vs. wetland count — which are genuinely different measures.
- `bioreactor_sat_buffer` carries **two category series** (the original
  *Bioreactors and Saturated Buffers* and a *(Updated 2022)* re-count) that
  overlap in time.

So the pivot column is built from `practice_type` + a category-vintage token +
`unit`, which makes `(huc8_code, year, col)` unique. The vintage token is empty
for the base series and `upd2022` for the re-counted bioreactor series; the
`CATEGORY_TOKEN` map is asserted to cover every category so a new one fails loud.

In [3]:
bmp = pd.read_csv(f"{CLEAN}/bmp/iowa-nrs-bmp-huc8-clean.csv", dtype={"huc8_code": str})
bmp["year"] = bmp["year"].astype(int)

# Category-vintage discriminator: only the bioreactor re-count needs a token.
CATEGORY_TOKEN = {
    "Bioreactors and Saturated Buffers": "",
    "Bioreactors and Saturated Buffers (Updated 2022)": "upd2022",
    "340": "",
    "CREP Wetlands": "",
    "Erosion Control": "",
}
missing = set(bmp["category"].unique()) - set(CATEGORY_TOKEN)
assert not missing, f"unmapped BMP category: {missing}"

vintage = bmp["category"].map(CATEGORY_TOKEN)
unit_token = bmp["unit"].str.lower()  # 'acres' / 'number'
col = "bmp__" + bmp["practice_type"]
col = col.where(vintage == "", col + "__" + vintage)
bmp["col"] = col + "__" + unit_token

dup = bmp.duplicated(subset=["huc8_code", "year", "col"]).sum()
assert dup == 0, f"{dup} duplicate (huc8_code, year, col) rows — pivot would aggregate"

bmp_wide = bmp.pivot_table(
    index=HUC8_JOIN, columns="col", values="value", aggfunc="mean"
).reset_index()
bmp_wide.columns.name = None

bmp_cols = [c for c in bmp_wide.columns if c.startswith("bmp__")]
print(f"BMP wide: {len(bmp_wide):,} HUC-8 x year rows | {len(bmp_cols)} practice columns")
print("columns:", bmp_cols)
bmp_wide.head(3)

BMP wide: 733 HUC-8 x year rows | 6 practice columns
columns: ['bmp__bioreactor_sat_buffer__number', 'bmp__bioreactor_sat_buffer__upd2022__number', 'bmp__cover_crop__acres', 'bmp__crep_wetland__acres', 'bmp__crep_wetland__number', 'bmp__erosion_control__acres']


,huc8_code,year,bmp__bioreactor_sat_buffer__number,bmp__bioreactor_sat_buffer__upd2022__number,bmp__cover_crop__acres,bmp__crep_wetland__acres,bmp__crep_wetland__number,bmp__erosion_control__acres
0,07020009,2017,NaN,NaN,2714.68,NaN,NaN,NaN
1,07020009,2022,NaN,NaN,5020.00,NaN,NaN,NaN
2,07040008,2017,NaN,NaN,111.22,NaN,NaN,NaN


## Step 3: Left-join BMP onto land use, then save

Land use is the left table, so the output is exactly its HUC-12 × year grid with
the HUC-8 BMP block broadcast on `huc8_code` + `year`. The `(huc12_code, year)`
grain is re-asserted after the join, and the row count must equal the land-use
input (a `left` join to a unique-keyed right table can neither add nor drop rows).

In [4]:
df = lu.merge(bmp_wide, on=HUC8_JOIN, how="left")

assert len(df) == len(lu), f"row count changed on join: {len(df)} vs {len(lu)}"
assert not df.duplicated(subset=KEY).any(), "output grain violated: duplicate (huc12_code, year)"

# Order columns: keys first (huc12, derived huc8, year), then land use, then BMP.
lu_cols = [c for c in lu.columns if c not in ("huc12_code", "huc8_code", "year")]
df = df[["huc12_code", "huc8_code", "year"] + lu_cols + bmp_cols]
df = df.sort_values(KEY).reset_index(drop=True)

print(f"Final shape: {df.shape}")
print(f"HUC-12 x year rows: {len(df):,} | distinct HUC-12s: {df['huc12_code'].nunique():,} | "
      f"year range: {df['year'].min()}-{df['year'].max()}")

any_bmp = df[bmp_cols].notna().any(axis=1)
print(f"\nHUC-12-years with any BMP value: {any_bmp.sum():,} / {len(df):,}")
print("BMP coverage by year (HUC-12-years with any BMP value):")
print(df.groupby("year").apply(lambda d: d[bmp_cols].notna().any(axis=1).sum()))

Final shape: (18854, 18)
HUC-12 x year rows: 18,854 | distinct HUC-12s: 1,714 | year range: 2015-2025

HUC-12-years with any BMP value: 13,596 / 18,854
BMP coverage by year (HUC-12-years with any BMP value):
year
2015    1695
2016    1695
2017    1713
2018    1695
2019    1695
2020    1695
2021    1695
2022    1713
2023       0
2024       0
2025       0
dtype: int64


In [5]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 18,854 rows x 18 cols -> ../../data/03a_merge_primary/huc12-landuse-bmp.csv
